# LESSON 5.3: The Fourier Slice Theorem
## Image Reconstruction from Projections

In this lesson:
- The Fourier Slice Theorem (Projection-Slice Theorem)
- Connecting projections to the 2-D Fourier Transform
- Why this theorem is the foundation of CT reconstruction
- Visual demonstration and verification
- Implications for reconstruction algorithms

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.transform import radon
from skimage.data import shepp_logan_phantom
from skimage.transform import rescale

## 1. The Fourier Slice Theorem

The **Fourier Slice Theorem** (also known as the **Central Slice Theorem** or **Projection-Slice Theorem**) is the fundamental theoretical basis for CT reconstruction.

### Statement:

$$\boxed{G(\omega, \theta) = F(\omega\cos\theta, \omega\sin\theta)}$$

Where:
- $G(\omega, \theta)$ = 1-D Fourier Transform of the projection $g(\rho, \theta)$ with respect to $\rho$
- $F(u, v)$ = 2-D Fourier Transform of the image $f(x, y)$
- $\omega$ = frequency variable in the projection direction
- $\theta$ = projection angle

### In plain words:
> The 1-D Fourier Transform of a projection at angle $\theta$ gives us a **slice** (a line through the origin at angle $\theta$) of the 2-D Fourier Transform of the image.

### Mathematical Derivation:

Starting from the Radon Transform:
$$g(\rho, \theta) = \int_{-\infty}^{\infty} \int_{-\infty}^{\infty} f(x, y) \, \delta(x\cos\theta + y\sin\theta - \rho) \, dx \, dy$$

Taking the 1-D Fourier Transform with respect to $\rho$:
$$G(\omega, \theta) = \int_{-\infty}^{\infty} g(\rho, \theta) \, e^{-j2\pi\omega\rho} \, d\rho$$

Substituting and simplifying:
$$G(\omega, \theta) = \int\int f(x,y) \, e^{-j2\pi\omega(x\cos\theta + y\sin\theta)} \, dx \, dy$$

Let $u = \omega\cos\theta$ and $v = \omega\sin\theta$:
$$G(\omega, \theta) = \int\int f(x,y) \, e^{-j2\pi(ux + vy)} \, dx \, dy = F(u, v)$$

In [ ]:
# Visual explanation of the Fourier Slice Theorem
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Create a simple phantom
size = 256
phantom = shepp_logan_phantom()
phantom = rescale(phantom, 256/400, anti_aliasing=True)
# Ensure it's exactly 256x256
phantom = phantom[:size, :size]

# 2D FFT of the phantom
F2D = np.fft.fftshift(np.fft.fft2(phantom))
magnitude_2d = np.log1p(np.abs(F2D))

# Show the image
axes[0].imshow(phantom, cmap='gray')
axes[0].set_title('Image f(x, y)', fontsize=13)
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')

# Show 2D Fourier Transform with slice lines
axes[1].imshow(magnitude_2d, cmap='viridis', extent=[-size//2, size//2, size//2, -size//2])
axes[1].set_title('2-D FT: F(u, v) with Slices', fontsize=13)
axes[1].set_xlabel('u')
axes[1].set_ylabel('v')

# Draw slice lines
angles_show = [0, 45, 90, 135]
colors = ['red', 'yellow', 'cyan', 'lime']
for angle, color in zip(angles_show, colors):
    rad = np.radians(angle)
    length = size // 2
    axes[1].plot([-length*np.cos(rad), length*np.cos(rad)],
               [-length*np.sin(rad), length*np.sin(rad)],
               color=color, linewidth=2, label=f'$\\theta$ = {angle}°')
axes[1].legend(fontsize=9, loc='upper right')

# Show 1D FFTs of projections (slices)
theta_array = np.array(angles_show, dtype=float)
for angle, color in zip(angles_show, colors):
    # Compute projection at this angle
    projection = radon(phantom, theta=np.array([angle], dtype=float), circle=False)
    proj_1d = projection[:, 0]
    
    # 1D FFT of the projection
    G1D = np.fft.fftshift(np.fft.fft(proj_1d))
    freqs = np.fft.fftshift(np.fft.fftfreq(len(proj_1d)))
    
    axes[2].plot(freqs, np.log1p(np.abs(G1D)), color=color, linewidth=1.5,
               label=f'$\\theta$ = {angle}°')

axes[2].set_title('1-D FT of Projections G($\\omega$, $\\theta$)', fontsize=13)
axes[2].set_xlabel('Frequency $\\omega$')
axes[2].set_ylabel('log(1 + |G|)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('The Fourier Slice Theorem: Projections → Slices of 2-D FT',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Each colored line in the 2D FT (center) corresponds to the 1D FT")
print("of the projection at that angle (right plot).")

## 2. Verifying the Fourier Slice Theorem

Let's numerically verify that the 1-D FT of a projection equals a slice of the 2-D FT.

In [ ]:
# Numerical verification of the Fourier Slice Theorem
size = 256

# Create a test image
image = np.zeros((size, size))
from skimage.draw import disk, ellipse
rr, cc = ellipse(128, 128, 60, 40)
valid = (rr >= 0) & (rr < size) & (cc >= 0) & (cc < size)
image[rr[valid], cc[valid]] = 1.0

# Method 1: 1-D FT of projection at theta=0
projection_0 = radon(image, theta=np.array([0.0]), circle=False)
G_from_projection = np.fft.fftshift(np.fft.fft(projection_0[:, 0]))

# Method 2: Slice of 2-D FT at theta=0 (horizontal line through center)
F2D = np.fft.fftshift(np.fft.fft2(image))
# For theta=0, the slice is along the v=0 line (horizontal through center)
center = size // 2

# Need to interpolate since projection may have different size
slice_from_2dft = F2D[center, :]  # horizontal slice

# Resample to match projection size
proj_size = len(projection_0[:, 0])
freqs_proj = np.linspace(-0.5, 0.5, proj_size)
freqs_2d = np.linspace(-0.5, 0.5, size)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Show the magnitudes
axes[0].plot(np.log1p(np.abs(G_from_projection)), 'b-', linewidth=2,
            label='1-D FT of projection')
axes[0].set_title('Method 1: |FT{g(ρ, 0°)}|', fontsize=12)
axes[0].set_xlabel('Frequency index')
axes[0].set_ylabel('log(1 + |G|)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(np.log1p(np.abs(slice_from_2dft)), 'r-', linewidth=2,
            label='Slice of 2-D FT')
axes[1].set_title('Method 2: |F(u, 0)| (slice of 2D FT)', fontsize=12)
axes[1].set_xlabel('Frequency index')
axes[1].set_ylabel('log(1 + |F|)')
axes[1].grid(True, alpha=0.3)

# Overlay for comparison (need to interpolate)
from scipy.interpolate import interp1d
interp_func = interp1d(np.linspace(0, 1, len(slice_from_2dft)),
                       np.abs(slice_from_2dft), kind='linear')
slice_resampled = interp_func(np.linspace(0, 1, len(G_from_projection)))

axes[2].plot(np.log1p(np.abs(G_from_projection)), 'b-', linewidth=2,
            label='1-D FT of projection', alpha=0.7)
axes[2].plot(np.log1p(slice_resampled), 'r--', linewidth=2,
            label='Slice of 2-D FT', alpha=0.7)
axes[2].set_title('Comparison: Both Methods', fontsize=12)
axes[2].set_xlabel('Frequency index')
axes[2].set_ylabel('log(1 + magnitude)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Fourier Slice Theorem Verification', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("The two curves match closely, confirming the Fourier Slice Theorem!")
print("(Small differences arise from discretization and interpolation.)")

## 3. Filling the Fourier Space

### Key Insight:
- Each projection at angle $\theta$ gives us **one radial line** in the 2-D Fourier domain
- With **enough projections** at different angles, we can fill the entire 2-D Fourier space
- Then we can apply the **inverse 2-D FT** to reconstruct the image

### The Problem:
- Projections give us data on **radial lines** (polar grid)
- The inverse FFT requires data on a **Cartesian grid**
- We need to **interpolate** from polar to Cartesian — this introduces errors
- The data density is **non-uniform**: denser near the origin, sparser at high frequencies

In [ ]:
# Visualize how projections fill the Fourier space
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

num_projections_list = [4, 18, 45, 180]

for i, n_proj in enumerate(num_projections_list):
    angles = np.linspace(0, 180, n_proj, endpoint=False)
    
    # Draw radial lines in Fourier space
    for angle in angles:
        rad = np.radians(angle)
        x = np.array([-np.cos(rad), np.cos(rad)]) * 1.0
        y = np.array([-np.sin(rad), np.sin(rad)]) * 1.0
        axes[i].plot(x, y, 'b-', linewidth=0.8, alpha=0.6)
    
    # Draw the boundary circle
    theta_circle = np.linspace(0, 2*np.pi, 100)
    axes[i].plot(np.cos(theta_circle), np.sin(theta_circle), 'r-', linewidth=1.5)
    axes[i].plot(0, 0, 'ro', markersize=5)
    
    axes[i].set_xlim([-1.2, 1.2])
    axes[i].set_ylim([-1.2, 1.2])
    axes[i].set_aspect('equal')
    axes[i].set_title(f'{n_proj} Projections', fontsize=12)
    axes[i].set_xlabel('u')
    if i == 0:
        axes[i].set_ylabel('v')
    axes[i].grid(True, alpha=0.2)

plt.suptitle('Fourier Space Coverage: Each Projection Adds One Radial Line',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("With few projections (left): Large gaps between radial lines → poor reconstruction.")
print("With many projections (right): Dense coverage → good reconstruction.")
print("\nNote: Coverage is always denser near the center (low frequencies)")
print("and sparser at the edges (high frequencies).")

## 4. Direct Fourier Reconstruction

Based on the Fourier Slice Theorem, a natural reconstruction approach is:

1. Compute the 1-D FT of each projection
2. Place each 1-D FT as a radial line in the 2-D Fourier space
3. Interpolate from polar to Cartesian grid
4. Apply inverse 2-D FFT

This is called **Direct Fourier Reconstruction**.

### Limitations:
- Interpolation in Fourier space introduces artifacts
- The non-uniform sampling density requires **density compensation**
- In practice, **Filtered Back Projection** (next lessons) is preferred

In [ ]:
# Direct Fourier Reconstruction (simplified demonstration)
size = 128  # smaller for speed
image = np.zeros((size, size))
rr, cc = disk((64, 64), 30)
image[rr, cc] = 1.0
rr, cc = disk((50, 50), 10)
image[rr, cc] = 0.5

n_projections = 180
theta = np.linspace(0, 180, n_projections, endpoint=False)

# Compute sinogram
sinogram = radon(image, theta=theta, circle=False)
n_detector = sinogram.shape[0]

# Direct Fourier Reconstruction
# Step 1: 1-D FFT of each projection
sinogram_ft = np.fft.fftshift(np.fft.fft(sinogram, axis=0), axes=0)

# Step 2: Place on 2-D Cartesian grid using interpolation
fourier_2d = np.zeros((size, size), dtype=complex)
center = size // 2

freqs = np.fft.fftshift(np.fft.fftfreq(n_detector))

for i, angle_deg in enumerate(theta):
    angle_rad = np.radians(angle_deg)
    for j, freq in enumerate(freqs):
        # Map polar (freq, angle) to Cartesian (u, v)
        u = freq * np.cos(angle_rad)
        v = freq * np.sin(angle_rad)
        
        # Convert to grid indices
        ui = int(round(u * size)) + center
        vi = int(round(v * size)) + center
        
        if 0 <= ui < size and 0 <= vi < size:
            fourier_2d[vi, ui] = sinogram_ft[j, i]

# Step 3: Inverse 2-D FFT
reconstruction = np.real(np.fft.ifft2(np.fft.ifftshift(fourier_2d)))

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(image, cmap='gray')
axes[0].set_title('Original Image', fontsize=12)

axes[1].imshow(sinogram, cmap='hot', aspect='auto')
axes[1].set_title('Sinogram', fontsize=12)

axes[2].imshow(np.log1p(np.abs(fourier_2d)), cmap='viridis')
axes[2].set_title('Filled Fourier Space', fontsize=12)

axes[3].imshow(reconstruction, cmap='gray')
axes[3].set_title('Direct Fourier\nReconstruction', fontsize=12)

plt.suptitle('Direct Fourier Reconstruction (Simplified)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Direct Fourier Reconstruction suffers from interpolation artifacts.")
print("Filtered Back Projection (Lesson 5.5) gives much better results.")

## 5. The Density Problem

The radial sampling in Fourier space creates a **non-uniform density**:

- **Near the origin** (low frequencies): Many radial lines are close together → **oversampled**
- **Far from origin** (high frequencies): Radial lines spread apart → **undersampled**

The density of samples at frequency $\omega$ is proportional to $\frac{1}{|\omega|}$.

This means low frequencies are **overrepresented**, leading to **blurring** in simple back projection.

### Solution:
Weight the data by $|\omega|$ to compensate — this is the basis of **filtered** back projection!

In [ ]:
# Demonstrate the density problem
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Show sampling density in Fourier space
n_proj = 36
angles = np.linspace(0, 180, n_proj, endpoint=False)
n_samples = 100

all_u = []
all_v = []

for angle in angles:
    rad = np.radians(angle)
    freqs = np.linspace(-0.5, 0.5, n_samples)
    u = freqs * np.cos(rad)
    v = freqs * np.sin(rad)
    all_u.extend(u)
    all_v.extend(v)
    axes[0].plot(u, v, 'b.', markersize=1, alpha=0.3)

axes[0].set_xlim([-0.55, 0.55])
axes[0].set_ylim([-0.55, 0.55])
axes[0].set_aspect('equal')
axes[0].set_title(f'Sample Points in Fourier Space\n({n_proj} projections)', fontsize=12)
axes[0].set_xlabel('u')
axes[0].set_ylabel('v')
axes[0].grid(True, alpha=0.3)

# Show the density as a function of radius
# Density ∝ 1/|ω| (fewer samples per area at high frequencies)
omega = np.linspace(0.01, 0.5, 100)
density = 1.0 / omega  # proportional to 1/|omega|
density_corrected = np.ones_like(omega)  # after weighting by |omega|

axes[1].plot(omega, density / density.max(), 'r-', linewidth=2,
            label='Without correction: $\\propto 1/|\\omega|$')
axes[1].plot(omega, density_corrected, 'b-', linewidth=2,
            label='With $|\\omega|$ weighting: uniform')
axes[1].fill_between(omega, 0, density / density.max(), alpha=0.2, color='red')
axes[1].set_title('Sampling Density vs Frequency', fontsize=12)
axes[1].set_xlabel('Frequency $|\\omega|$')
axes[1].set_ylabel('Relative Density')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('The Non-Uniform Density Problem in Fourier Space',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Low frequencies are oversampled (dense near center).")
print("High frequencies are undersampled (sparse at edges).")
print("\nSolution: Multiply by |ω| to get uniform density → Filtered Back Projection!")

## 6. From the Fourier Slice Theorem to Filtered Back Projection

Starting from the inverse 2-D Fourier Transform in polar coordinates:

$$f(x, y) = \int_0^{\pi} \left[ \int_{-\infty}^{\infty} G(\omega, \theta) \, |\omega| \, e^{j2\pi\omega\rho} \, d\omega \right] d\theta$$

where $\rho = x\cos\theta + y\sin\theta$.

The inner integral is the **filtered projection** (1-D inverse FT of $G \cdot |\omega|$):

$$\tilde{g}(\rho, \theta) = \int_{-\infty}^{\infty} G(\omega, \theta) \, |\omega| \, e^{j2\pi\omega\rho} \, d\omega$$

The outer integral is the **back projection** (smearing the filtered projection back):

$$f(x, y) = \int_0^{\pi} \tilde{g}(x\cos\theta + y\sin\theta, \theta) \, d\theta$$

This gives us the **Filtered Back Projection (FBP)** algorithm:

$$\boxed{f(x,y) = \int_0^{\pi} \tilde{g}(x\cos\theta + y\sin\theta, \theta) \, d\theta}$$

The $|\omega|$ term is the **ramp filter** — it compensates for the density problem!

In [ ]:
# Show the ramp filter |omega|
omega = np.linspace(-0.5, 0.5, 256)

# Different filters used in CT reconstruction
ramp = np.abs(omega)
shepp_logan_filter = np.abs(omega) * np.sinc(omega / (2 * 0.5))
cosine_filter = np.abs(omega) * np.cos(np.pi * omega / (2 * 0.5))
hamming_filter = np.abs(omega) * (0.54 + 0.46 * np.cos(np.pi * omega / 0.5))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Frequency domain
axes[0].plot(omega, ramp, 'b-', linewidth=2, label='Ramp (Ram-Lak)')
axes[0].plot(omega, shepp_logan_filter, 'r-', linewidth=2, label='Shepp-Logan')
axes[0].plot(omega, cosine_filter, 'g-', linewidth=2, label='Cosine')
axes[0].plot(omega, hamming_filter, 'm-', linewidth=2, label='Hamming')
axes[0].set_title('Reconstruction Filters (Frequency Domain)', fontsize=12)
axes[0].set_xlabel('Frequency $\\omega$')
axes[0].set_ylabel('$H(\\omega)$')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Show the concept: unfiltered vs filtered projection
phantom = shepp_logan_phantom()
phantom = rescale(phantom, 0.5, anti_aliasing=True)
theta_single = np.array([0.0])
projection = radon(phantom, theta=theta_single, circle=True)
proj = projection[:, 0]

# Apply ramp filter
n = len(proj)
freqs = np.fft.fftfreq(n)
ramp_filter = np.abs(freqs)
proj_fft = np.fft.fft(proj)
proj_filtered = np.real(np.fft.ifft(proj_fft * ramp_filter * 2))

axes[1].plot(proj / proj.max(), 'b-', linewidth=2, label='Original projection')
axes[1].plot(proj_filtered / np.abs(proj_filtered).max(), 'r-', linewidth=2,
            label='After ramp filtering')
axes[1].set_title('Projection Before and After Ramp Filtering', fontsize=12)
axes[1].set_xlabel('Detector position $\\rho$')
axes[1].set_ylabel('Normalized value')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('From Fourier Slice Theorem to Filtered Back Projection',
            fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("The ramp filter |ω| enhances edges and suppresses the blurring")
print("caused by the 1/|ω| density distribution in Fourier space.")

## Summary

What we learned:
1. The **Fourier Slice Theorem**: The 1-D FT of a projection at angle $\theta$ equals a radial slice of the 2-D FT at the same angle
2. Each projection fills **one radial line** in the 2-D Fourier space
3. With enough projections, we can fill the entire Fourier space and reconstruct the image
4. **Direct Fourier Reconstruction** suffers from interpolation artifacts
5. The **non-uniform density** problem: low frequencies are oversampled, high frequencies are undersampled
6. The $|\omega|$ (ramp) filter compensates for this density imbalance
7. This leads to the **Filtered Back Projection** algorithm, the most common CT reconstruction method